# Convert pka data from pdfs to csvs for use in pipelines

In [4]:
import re

import pathlib
from pathlib import Path
import pdfplumber

from rdkit import Chem
from rdkit.Chem import AllChem
import pubchempy as pcp
import pdfplumber

import pandas as pd
import numpy as np
from sklearn.cluster import KMeans

In [5]:
working_dir = 'pkas'

In [6]:
# define paths
HERE = Path(pathlib.Path.cwd())
DATA = HERE / f"data_{working_dir}"
DATA.mkdir(parents=True, exist_ok=True)

In [7]:
# acid classes for automatic name repair

ACID_CLASSES = {
    "dicarboxylic acids": "acid",
    "carboxylic acids": "acid",
    "unsaturated acids": "acid",
    "alicyclic dicarboxylic acids": "acid",
    "amino acids": "acid",
    "aliphatic acids": "acid",
    "aromatic acids": "acid",
    "dicarboxylic acids, unsaturated": "acid",
    # other categories can be added
}

def detect_table_title(line: str):
    clean = line.lower().strip()
    for title in ACID_CLASSES:
        if title in clean:
            return title
    return None

def repair_name(name, current_class):
    """
    Append acid suffix based on table heading if appropriate.
    """
    if not current_class:
        return name

    suffix = ACID_CLASSES.get(current_class)
    if suffix is None:
        return name

    # don't append "acid" in iall cases
    if re.search(r"(acid|ate|one|ol|ene|ide|ium|oxide|amine|thiol)$", name.lower()):
        return name

    return f"{name} {suffix}"

# Dynamic column splitting (left/right) for multi-column PDFs
def find_dynamic_column_split(page, n_clusters=2):
    xs = np.array([[c["x0"]] for c in page.chars])
    if len(xs) < 10:
        return page.width / 2  # fallback
    kmeans = KMeans(n_clusters=n_clusters, n_init=10).fit(xs)
    centers = sorted([c[0] for c in kmeans.cluster_centers_])
    split = (centers[0] + centers[1]) / 2
    return split

def split_columns_dynamic(page):
    width, height = page.width, page.height
    split = find_dynamic_column_split(page)
    left_box = (0, 0, split, height)
    right_box = (split, 0, width, height)
    left_text = page.crop(left_box).extract_text()
    right_text = page.crop(right_box).extract_text()
    return [left_text, right_text]

# Line parser for compound + pKa extraction
PKA_RE = re.compile(r"-?\d+\.\d+\*?")

def parse_line(line, current_class):
    """
    Extract name, pKa(s), ref(s) from one line.
    Handles multiple pKas and multiple refs.
    """
    line = line.replace("–", "-").replace("—", "-")
    line = re.sub(r"\s+", " ", line).strip()

    # Extract pKas
    pka_vals = PKA_RE.findall(line)
    if not pka_vals:
        return None
    pkas = [float(v.replace("*", "")) for v in pka_vals]

    # Remove pKas from line
    line_no_pka = PKA_RE.sub("", line).strip()

    # Extract references (numbers at end, multiple separated by comma)
    ref_match = re.search(r"(\d+(?:\s*,\s*\d+)*)\s*$", line_no_pka)
    if ref_match:
        ref_str = ref_match.group(1)
        refs = [int(x.strip()) for x in ref_str.split(",")]
        # Remove refs from name
        line_no_pka = re.sub(rf"{re.escape(ref_str)}\s*$", "", line_no_pka).strip()
    else:
        refs = [None]

    # Align refs to pkas
    if len(refs) == 1:
        refs = refs * len(pkas)
    elif len(refs) != len(pkas):
        refs = [refs[0]] * len(pkas)  # fallback: replicate first ref

    # Clean up trailing commas, spaces
    name = line_no_pka.strip().strip(",").replace("  ", " ")
    name = repair_name(name, current_class)

    # Replace '=' at the end of formula with '2+'
    name = re.sub(r"\s*=$", " 2+", name)

    # Reattach trailing charges
    name = re.sub(r"\s+([+-]\d*)$", r"\1", name)

    # Generate output entries for each pKa
    entries = []
    for i, (pka, ref) in enumerate(zip(pkas, refs), start=1):
        entries.append({
            "name": name,
            "site": i,
            "pKa": pka,
            "ref": ref
        })

    return entries

# Main extraction function
def extract_pka_data(pdf_path):
    rows = []
    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            col_texts = split_columns_dynamic(page)
            current_class = None

            for text in col_texts:
                if not text:
                    continue
                for line in text.split("\n"):
                    new_class = detect_table_title(line)
                    if new_class:
                        current_class = new_class
                        continue
                    entry = parse_line(line, current_class)
                    if entry:
                        for e in entry:
                            e["page"] = page_num
                            rows.append(e)
    return rows

In [8]:
# Convert to long
def rows_to_long_df(rows):
    df = pd.DataFrame(rows)
    df["name"] = df["name"].str.strip()
    return df[["name", "site", "pKa", "ref", "page"]]

## Lookup compund names on Pubchem to get SMILES.

In [10]:
rows = extract_pka_data(f"{DATA}/pka-compilation-williams.pdf")

In [11]:
df_long = rows_to_long_df(rows)
df_long.head()

,name,site,pKa,ref,page
0,AgOH,1,3.96,4.0,2
1,Al(OH)3,1,11.20,28.0,2
2,As(OH)3,1,9.22,28.0,2
3,"H3AsO4 ,",1,2.22,28.0,2
4,"H3AsO4 ,",2,7.00,28.0,2


In [ ]:
# def clean_formula(formula: str) -> str:
#     """
#     Remove trailing characters or annotations that are unlikely to be
#     part of a chemical formula, while keeping valid chemical symbols,
#     parentheses for groupings, charges, +, -, and equals signs.
#     """
#     formula = formula.strip()                 # remove leading/trailing spaces
#     formula = re.sub(r"[,*\s]+$", "", formula)  # remove trailing commas, asterisks, spaces
    
#     # Remove parenthetical annotations at the end if they contain non-chemical characters
#     # e.g. "HClO4 (70%)" -> "HClO4"
#     formula = re.sub(r"\s*\([^A-Za-z0-9()+\-=\s]*\)\s*$", "", formula)
    
#     return formula

def clean_formula(formula: str) -> str:
    """
    Clean up chemical formulae by removing trailing non-chemical characters
    like commas, asterisks, extra spaces, and parenthetical annotations
    that are unlikely part of the formula.
    """
    formula = formula.strip()
    formula = re.sub(r"[,*\s]+$", "", formula)
    formula = re.sub(r"\s*\([^A-Za-z0-9()+\-=\s]*\)\s*$", "", formula)
    return formula

def formula_to_smiles(formula: str):
    """
    Attempt to convert a chemical formula to SMILES using PubChem search.
    Returns None if no results.
    """
    try:
        compounds = pcp.get_compounds(formula, 'formula')
        if compounds:
            return compounds[0].canonical_smiles
        return None
    except Exception:
        return None

def name_to_smiles(name: str):
    """
    Convert a common chemical name to SMILES using PubChemPy.
    Returns None if not found.
    """
    try:
        compounds = pcp.get_compounds(name, 'name')
        if compounds:
            return compounds[0].canonical_smiles
        return None
    except Exception:
        return None

def flexible_name_lookup(name: str):
    """
    Try multiple variants of a chemical name to maximize PubChem hits.
    """
    # Base attempt
    smi = name_to_smiles(name)
    if smi:
        return smi

    # Variant 1: replace colons with commas
    variant1 = re.sub(r":", ",", name)
    smi = name_to_smiles(variant1)
    if smi:
        return smi

    # Variant 2: reorder locants if pattern matches "X meso-Y ..." -> "meso-Y-X ..."
    match = re.match(r"(\d+)\s+(meso-\d+.*)", name, re.I)
    if match:
        variant2 = f"{match.group(2)} {match.group(1)}"
        smi = name_to_smiles(variant2)
        if smi:
            return smi

    # Variant 3: remove hyphens
    variant3 = re.sub(r"[-]", " ", name)
    smi = name_to_smiles(variant3)
    return smi


In [ ]:
# def generate_smiles(name: str):
#     """
#     Decide if the input is a formula or a name and return SMILES.
#     Applies cleaning and flexible name lookup.
#     """
#     cleaned = clean_formula(name)

#     # If it looks like a formula (letters, numbers, parentheses, +, -, =)
#     if re.match(r"^[A-Za-z0-9()+=-]+$", cleaned):
#         smi = formula_to_smiles(cleaned)
#         if smi:
#             return smi

#     # Otherwise, treat it as a common name (with flexible lookup)
#     return flexible_name_lookup(cleaned)

In [ ]:
def robust_generate_smiles(identifier: str):
    """
    Given a single identifier (formula or common name), return a SMILES string if possible.
    Applies cleaning, flexible variants, and multiple PubChem identifier types.
    """
    import re
    import pubchempy as pcp

    def clean_input(s: str) -> str:
        s = s.strip()
        s = re.sub(r"[,*\s]+$", "", s)  # trailing commas, asterisks, spaces
        s = re.sub(r"\s*\([^A-Za-z0-9()+\-=\s]*\)\s*$", "", s)  # trailing annotations
        return s

    def try_pubchem(query: str, types=("formula", "name", "synonym", "iupac_name")):
        for id_type in types:
            try:
                compounds = pcp.get_compounds(query, id_type)
                if compounds:
                    return compounds[0].canonical_smiles
            except Exception:
                continue
        return None

    def flexible_name_lookup(name: str):
        # base
        smi = try_pubchem(name)
        if smi:
            return smi

        # colons → commas
        variant1 = re.sub(r":", ",", name)
        smi = try_pubchem(variant1)
        if smi:
            return smi

        # reorder locants like "2 meso-1:2-Dibromosuccinic acid"
        match = re.match(r"(\d+)\s+(meso-\d+.*)", name, re.I)
        if match:
            variant2 = f"{match.group(2)} {match.group(1)}"
            smi = try_pubchem(variant2)
            if smi:
                return smi

        # remove hyphens
        variant3 = re.sub(r"[-]", " ", name)
        smi = try_pubchem(variant3)
        return smi

    # --- main logic ---
    identifier = clean_input(identifier)

    # formula-like pattern: letters, numbers, parentheses, +, -, =
    if re.match(r"^[A-Za-z0-9()+=-]+$", identifier):
        smi = try_pubchem(identifier)
        if smi:
            return smi
        # fallback to treat it as a flexible name too
        return flexible_name_lookup(identifier)

    # otherwise treat as common name
    return flexible_name_lookup(identifier)

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm

# Cache to avoid repeated lookups
smiles_cache = {}

def lookup_smiles(name):
    """
    Wrapper that caches results and handles exceptions.
    """
    if name in smiles_cache:
        return smiles_cache[name]

    try:
        # smi = robust_generate_smiles(name) # use generate_smiles to less robust but faster queries.
        smi = robust_generate_smiles(name)
    except Exception:
        smi = None

    smiles_cache[name] = smi
    return smi


In [ ]:
df_long_ = df_long.head(30) # prototyping

In [19]:
def parallel_smiles_lookup(names, max_workers=10):
    results = [None] * len(names)
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_idx = {executor.submit(lookup_smiles, name): i for i, name in enumerate(names)}
        
        for future in tqdm(as_completed(future_to_idx), total=len(future_to_idx), desc="Generating SMILES"):
            i = future_to_idx[future]
            results[i] = future.result()
    
    return results


In [20]:
# Apply to DataFrame
df_long_["SMILES"] = parallel_smiles_lookup(df_long_["name"].tolist(), max_workers=10)

Generating SMILES:   0%|          | 0/30 [00:00<?, ?it/s]

/tmp/ipykernel_84138/3990634788.py:117: PubChemPyDeprecationWarning: canonical_smiles is deprecated: Use connectivity_smiles instead
  return compounds[0].canonical_smiles
/tmp/ipykernel_84138/2343721846.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_long_["SMILES"] = parallel_smiles_lookup(df_long_["name"].tolist(), max_workers=10)


In [21]:
df_long_

,name,site,pKa,ref,page,SMILES
0,AgOH,1,3.96,4.0,2,[OH-].[Ag+]
1,Al(OH)3,1,11.20,28.0,2,None
2,As(OH)3,1,9.22,28.0,2,O[As](O)O
3,"H3AsO4 ,",1,2.22,28.0,2,O[As](=O)(O)O
4,"H3AsO4 ,",2,7.00,28.0,2,O[As](=O)(O)O
5,"H3AsO4 ,",3,13.00,28.0,2,O[As](=O)(O)O
6,H2AsO4-,1,6.98,77.0,2,O[As](=O)(O)[O-]
7,HAsO4*,1,11.53,77.0,2,O[As](=O)([O-])[O-]
8,H3AsO,1,9.22,3.0,2,O[AsH2]
9,H3BO3,1,9.23,28.0,2,B(O)(O)O


In [ ]:
df_long.to_csv(f"{DATA}/clean_pKa_long.csv", index=False)